# Titanic Dataset — First Exploratory Data Analysis (EDA)

**Goal:** Set up my Python data science toolkit (pandas, NumPy) and "listen" to the Titanic dataset before doing any modeling — understand its shape, quality, and quirks.

Dataset: [Titanic - Machine Learning from Disaster](https://www.kaggle.com/competitions/titanic) (Kaggle)

## 1. Setup — Import libraries

In [1]:
import pandas as pd
import numpy as np

# Quick sanity check on versions
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.2.2
numpy version: 2.0.2


## 2. Load the dataset

I downloaded `train.csv` from the Kaggle "Titanic - Machine Learning from Disaster" competition and placed it in the same folder as this notebook.

In [2]:
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. First look — shape, structure, and summary statistics

In [3]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.info()

Rows: 891, Columns: 12
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df.describe(include='object')

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Dooley, Mr. Patrick",male,347082,G6,S
freq,1,577,7,4,644


## 4. Missing values

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False)

,missing_count,missing_pct
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


## 5. Categorical vs. numerical columns

In [7]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


> Note: `Survived` and `Pclass` are stored as integers but are really **categorical** (labels/ordinal classes),
> not continuous quantities — worth remembering before modeling.

## 6. My first data story

- The dataset has **891 rows and 12 columns**, one row per passenger.
- Three columns have missing values: **`Cabin`** (77% missing — mostly unusable as-is), **`Age`** (~20% missing — fixable with imputation), and **`Embarked`** (just 2 missing rows).
- **Numerical** columns: `PassengerId`, `Survived`, `Pclass`, `Age`, `SibSp`, `Parch`, `Fare` — though `Survived` and `Pclass` are really categorical labels stored as numbers, not true continuous values.
- **Categorical** columns: `Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`.
- `Age` and `Fare` are right-skewed (see the mean vs. median gap in `.describe()`), so I'll want to watch for outliers before modeling.
- Overall the dataset is small, mostly clean, and manageable — but `Cabin`'s heavy missingness and `Age`'s moderate missingness are the two things I'll need a strategy for before building any predictive model.